# 量化选股 ML 建模
## 基于 A 股全市场基本面数据的 Walk-Forward 回测

**Spec**: `ml_model_spec.md`  
**数据**: `model_data.csv` (39,617 行, 4,282 只股, 2020Q1~2022Q2)  
**模型池**: Logistic Regression / Random Forest / XGBoost / LightGBM  
**回测**: 5 窗口 Walk-Forward  

---

## 1. 环境准备与数据加载

In [ ]:
import os, json, warnings, joblib
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (roc_auc_score, log_loss, brier_score_loss,
                              accuracy_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)
from sklearn.model_selection import ParameterGrid
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

BASE_DIR = os.getcwd()
print(f"Working directory: {BASE_DIR}")

In [ ]:
# Load data
df = pd.read_csv('model_data.csv', encoding='utf-8')
df['Date'] = pd.to_datetime(df['Date']).dt.strftime('%Y/%m/%d')

print(f"Shape: {df.shape}")
print(f"Stocks: {df['Code'].nunique()}")
print(f"Date range: {df['Date'].min()} ~ {df['Date'].max()}")
print(f"\nColumns ({len(df.columns)}):")
for c in df.columns:
    print(f"  {c}")
df.head(3)

## 2. 数据清洗与特征工程

In [ ]:
# Rename columns for safe access
rename_map = {}
for c in df.columns:
    new = c.strip()
    if '(' in new: new = new.split('(')[0].strip()
    new = new.replace(' ', '_').replace('/', '_').replace('（', '').replace('）', '')
    rename_map[c] = new
df.rename(columns=rename_map, inplace=True)

# Drop rows without target
df = df.dropna(subset=['Next_Ret']).copy()
print(f"After cleaning: {len(df)} rows")

# Map raw columns to feature names
RAW_FACTORS = [
    '企业倍数_EV', '市净率PB', '市现率PCF', '市现率PCF_经营',
    '市盈率PE', '市盈率PE_扣非', '市销率PS', '股息率',
    'MV', '净利润同比增长率', '净资产同比增长率', '利润总额同比增长率',
    '基本每股收益同比增长率', '总资产同比增长率', '现金净流量同比增长率',
    '经营活动产生的现金流量净额同比增长率', '营业利润同比增长率',
    '营业总收入同比增长率', '营业收入同比增长率',
]

FEAT_COL_NAMES = [
    'EV_EBITDA', 'PB', 'PCF_NetCash', 'PCF_Operating',
    'PE_TTM', 'PE_TTM_Deducted', 'PS_TTM', 'Dividend_Yield',
    'MV', 'Profit_Growth', 'NetAsset_Growth', 'TotalProfit_Growth',
    'EPS_Growth', 'TotalAsset_Growth', 'NetCash_Growth',
    'OperatingCF_Growth', 'OperatingProfit_Growth',
    'Revenue1_Growth', 'Revenue2_Growth',
]

for raw, name in zip(RAW_FACTORS, FEAT_COL_NAMES):
    col_matches = [c for c in df.columns if raw in c or c == raw]
    if col_matches:
        df[name] = pd.to_numeric(df[col_matches[0]], errors='coerce')

# Winsorize 1%/99%
def winsorize_series(s, lo=0.01, hi=0.99):
    if s.dropna().empty: return s
    lo_v, hi_v = s.quantile(lo), s.quantile(hi)
    return s.clip(lo_v, hi_v)

for name in FEAT_COL_NAMES:
    if name in df.columns:
        df[name] = df.groupby('Date')[name].transform(winsorize_series)

df['MV_Log'] = np.log(df['MV'].clip(lower=0.01))

# Missing rate check
missing_pct = df[FEAT_COL_NAMES + ['MV_Log']].isnull().mean().sort_values(ascending=False)
print("Top 10 missing features:")
print(missing_pct.head(10))

In [ ]:
# Rank normalization function
RANK_TARGETS = [
    ('PE_TTM', 'R_PE', 'desc'), ('PB', 'R_PB', 'desc'),
    ('PS_TTM', 'R_PS', 'desc'), ('EV_EBITDA', 'R_EV', 'desc'),
    ('Profit_Growth', 'R_Profit_Growth', 'asc'),
    ('Revenue2_Growth', 'R_Revenue_Growth', 'asc'),
    ('Dividend_Yield', 'R_Dividend', 'asc'),
    ('MV', 'R_MV', 'asc'),
]

def apply_rank_normalize(data):
    data = data.copy()
    for src, dst, direction in RANK_TARGETS:
        if src not in data.columns: continue
        if direction == 'desc':
            data[dst] = data.groupby('Date')[src].rank(pct=True, ascending=False)
        else:
            data[dst] = data.groupby('Date')[src].rank(pct=True, ascending=True)
        data[dst] = data[dst].fillna(0.5)
    
    # Composite factors
    data['Value_Composite'] = data[['R_PE','R_PB','R_PS','R_EV']].mean(axis=1)
    data['Growth_Composite'] = data[['R_Profit_Growth','R_Revenue_Growth']].mean(axis=1)
    data['GARP_Signal'] = (data['Value_Composite'] + data['Growth_Composite']) / 2
    data['Quality_Score'] = data[['R_Profit_Growth','R_Revenue_Growth','R_Dividend']].mean(axis=1)
    return data

FEATURE_COLS = FEAT_COL_NAMES + ['MV_Log'] + \
              [d for _, d, _ in RANK_TARGETS] + \
              ['Value_Composite', 'Growth_Composite', 'GARP_Signal', 'Quality_Score']
print(f"Total features: {len(FEATURE_COLS)}")
print(FEATURE_COLS)

## 3. 构造标签

In [ ]:
df['Y'] = (df['Next_Ret'] > 0).astype(int)
print("Label distribution:")
print(df['Y'].value_counts())
print(f"\nPositive rate: {df['Y'].mean():.2%}")

# Visualize Next_Ret distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['Next_Ret'], bins=100, color='#4a7dff', edgecolor='white', alpha=0.8)
axes[0].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[0].set_title('Next_Ret Distribution')
axes[0].set_xlabel('Next Quarter Return')

df['Y'].value_counts().plot(kind='bar', ax=axes[1], color=['#e24b4a', '#639922'])
axes[1].set_title('Y Label Distribution')
axes[1].set_xticklabels(['Down (0)', 'Up (1)'], rotation=0)
plt.tight_layout()
plt.show()

## 4. Walk-Forward 训练与回测

In [ ]:
# Strategy parameters
STRATEGY = {
    'T_buy': 0.62, 'T_sell': 0.50, 'alpha': 0.5,
    'cap_stock': 0.05, 'N_max': 30, 'beta': 0.6,
}

# Walk-Forward windows
WINDOWS = [
    ('W1', '2020/3/31', '2020/12/31', '2021/3/31', '2021/6/30'),
    ('W2', '2020/3/31', '2021/3/31', '2021/6/30', '2021/9/30'),
    ('W3', '2020/3/31', '2021/6/30', '2021/9/30', '2021/12/31'),
    ('W4', '2020/3/31', '2021/9/30', '2021/12/31', '2022/3/31'),
    ('W5', '2020/3/31', '2021/12/31', '2022/3/31', '2022/6/30'),
]

print("Training function defined. Ready to run Walk-Forward loop.")

In [ ]:
# Model training functions
def train_logistic(X_train, y_train, X_val, y_val):
    best_score, best_model = -np.inf, None
    for C in [0.01, 0.1, 1.0, 10.0]:
        m = LogisticRegression(C=C, penalty='l2', solver='lbfgs', max_iter=2000, random_state=42)
        m.fit(X_train, y_train)
        score = roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])
        if score > best_score:
            best_score, best_model = score, m
    return best_model, best_score

def train_rf(X_train, y_train, X_val, y_val):
    best_score, best_model = -np.inf, None
    for depth in [5, 10, None]:
        for n in [100, 200]:
            m = RandomForestClassifier(n_estimators=n, max_depth=depth, min_samples_leaf=4,
                                      random_state=42, n_jobs=-1)
            m.fit(X_train, y_train)
            score = roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])
            if score > best_score:
                best_score, best_model = score, m
    return best_model, best_score

def train_xgb(X_train, y_train, X_val, y_val):
    best_score, best_model = -np.inf, None
    for lr in [0.05, 0.1]:
        for depth in [3, 5]:
            m = XGBClassifier(max_depth=depth, learning_rate=lr, n_estimators=200,
                              subsample=0.8, colsample_bytree=0.8,
                              reg_alpha=0.1, reg_lambda=1.0,
                              eval_metric='logloss', random_state=42, verbosity=0)
            m.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
            score = roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])
            if score > best_score:
                best_score, best_model = score, m
    return best_model, best_score

def train_lgb(X_train, y_train, X_val, y_val):
    best_score, best_model = -np.inf, None
    for lr in [0.05, 0.1]:
        for leaves in [15, 31]:
            m = LGBMClassifier(num_leaves=leaves, learning_rate=lr, n_estimators=200,
                               subsample=0.8, colsample_bytree=0.8,
                               reg_alpha=0.1, reg_lambda=1.0,
                               random_state=42, verbose=-1)
            m.fit(X_train, y_train, eval_set=[(X_val, y_val)])
            score = roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])
            if score > best_score:
                best_score, best_model = score, m
    return best_model, best_score

print("All training functions defined.")

In [ ]:
# Run Walk-Forward
all_results = []
model_metrics = []
model_list = []

for wname, train_start, train_end, val_end, test_end in WINDOWS:
    print(f"\n{'='*50}")
    print(f"{wname}: Train={train_start}~{train_end} | Val=~{val_end} | Test=~{test_end}")
    print('='*50)
    
    train_mask = (df['Date'] >= train_start) & (df['Date'] <= train_end)
    val_mask = (df['Date'] > train_end) & (df['Date'] <= val_end)
    test_mask = (df['Date'] > val_end) & (df['Date'] <= test_end)
    
    df_train = apply_rank_normalize(df[train_mask].copy())
    df_val = apply_rank_normalize(df[val_mask].copy())
    df_test = apply_rank_normalize(df[test_mask].copy())
    
    print(f"  Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")
    
    # Impute
    for _df in [df_train, df_val, df_test]:
        for c in FEATURE_COLS:
            if c in _df.columns:
                med = _df[c].median()
                _df[c] = _df[c].fillna(med if not np.isnan(med) else 0)
    
    X_train = df_train[FEATURE_COLS].values
    y_train = df_train['Y'].values
    X_val = df_val[FEATURE_COLS].values
    y_val = df_val['Y'].values
    
    # Train models
    print("  Training Logistic...")
    m_lr, auc_lr = train_logistic(X_train, y_train, X_val, y_val)
    print(f"    AUC={auc_lr:.4f}")
    
    print("  Training RandomForest...")
    m_rf, auc_rf = train_rf(X_train, y_train, X_val, y_val)
    print(f"    AUC={auc_rf:.4f}")
    
    print("  Training XGBoost...")
    m_xgb, auc_xgb = train_xgb(X_train, y_train, X_val, y_val)
    print(f"    AUC={auc_xgb:.4f}")
    
    print("  Training LightGBM...")
    m_lgb, auc_lgb = train_lgb(X_train, y_train, X_val, y_val)
    print(f"    AUC={auc_lgb:.4f}")
    
    models = {'Logistic': m_lr, 'RandomForest': m_rf, 'XGBoost': m_xgb, 'LightGBM': m_lgb}
    
    # Calibrate
    calibrators = {}
    for name, m in models.items():
        cal = CalibratedClassifierCV(m, method='sigmoid', cv='prefit')
        cal.fit(X_val, y_val)
        calibrators[name] = cal
    
    # Estimate b
    pos = df_train[df_train['Next_Ret'] > 0]['Next_Ret']
    neg = df_train[df_train['Next_Ret'] <= 0]['Next_Ret']
    b = max(pos.mean() / abs(neg.mean()), 0.5) if len(pos) > 0 and len(neg) > 0 else 1.0
    print(f"  b (odds) = {b:.3f}")
    
    # Backtest each model
    for mname, model in models.items():
        cal = calibrators[mname]
        p_cal = cal.predict_proba(df_test[FEATURE_COLS].values)[:, 1]
        df_test_tmp = df_test.copy()
        df_test_tmp['p_cal'] = p_cal
        
        candidates = df_test_tmp[df_test_tmp['p_cal'] > STRATEGY['T_buy']].copy()
        candidates = candidates.sort_values('p_cal', ascending=False).head(STRATEGY['N_max'])
        
        if len(candidates) > 0:
            candidates['f_kelly'] = candidates['p_cal'].apply(
                lambda p: max(0, (b * p - (1 - p)) / b))
            candidates['f_raw'] = (candidates['f_kelly'] * STRATEGY['alpha']).clip(upper=STRATEGY['cap_stock'])
            F_raw = candidates['f_raw'].sum()
            if F_raw > STRATEGY['beta']:
                candidates['f_final'] = candidates['f_raw'] * STRATEGY['beta'] / F_raw
            else:
                candidates['f_final'] = candidates['f_raw']
            port_ret = (candidates['f_final'] * candidates['Next_Ret']).sum()
        else:
            port_ret = 0.0
        
        all_results.append({
            'Window': wname, 'Model': mname, 'Portfolio_Return': port_ret,
            'N_Holdings': len(candidates),
        })
        
        # Validation metrics
        p_val = cal.predict_proba(X_val)[:, 1]
        model_metrics.append({
            'Window': wname, 'Model': mname,
            'AUC_Val': roc_auc_score(y_val, p_val),
            'LogLoss_Val': log_loss(y_val, p_val),
        })
        
        print(f"  {mname:15s}: return={port_ret:.4f}, holdings={len(candidates)}")

print(f"\nWalk-Forward complete. {len(all_results)} total results.")

## 5. 模型对比

In [ ]:
df_results = pd.DataFrame(all_results)
df_metrics = pd.DataFrame(model_metrics)

# Per-model summary
summary = df_results.groupby('Model').agg(
    Cumulative_Return=('Portfolio_Return', 'sum'),
    Mean_QR=('Portfolio_Return', 'mean'),
    Std_QR=('Portfolio_Return', 'std'),
    Avg_Holdings=('N_Holdings', 'mean'),
    Win_Rate=('Portfolio_Return', lambda x: (x > 0).mean()),
).reset_index()

summary['Sharpe'] = summary['Mean_QR'] / summary['Std_QR'].clip(lower=1e-6) * 2

# MaxDD
for i, row in summary.iterrows():
    returns = df_results[df_results['Model'] == row['Model']].sort_values('Window')['Portfolio_Return'].values
    cum = (1 + returns).cumprod()
    peak = np.maximum.accumulate(cum)
    summary.loc[i, 'MaxDD'] = abs((cum - peak) / peak).max()

summary['Calmar'] = summary['Mean_QR'] * 4 / summary['MaxDD'].clip(lower=1e-6)

print("=" * 60)
print("MODEL COMPARISON")
print("=" * 60)
summary[['Model', 'Cumulative_Return', 'Sharpe', 'MaxDD', 'Calmar', 'Win_Rate']].round(4)

In [ ]:
# Visualization: Cumulative returns
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'Logistic': '#7F77DD', 'RandomForest': '#378ADD', 'XGBoost': '#639922', 'LightGBM': '#1D9E75'}

for m in summary['Model']:
    mdata = df_results[df_results['Model'] == m].sort_values('Window')
    cum = (1 + mdata['Portfolio_Return']).cumprod()
    axes[0].plot(['Start'] + list(mdata['Window']), [1] + cum.tolist(),
                 'o-', label=m, color=colors[m], linewidth=2, markersize=6)

axes[0].axhline(1, color='gray', linestyle='--', linewidth=0.8)
axes[0].set_title('Walk-Forward Cumulative Returns')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Bar chart: Quarterly returns
x = np.arange(5)
w = 0.2
for i, m in enumerate(summary['Model']):
    mdata = df_results[df_results['Model'] == m].sort_values('Window')
    axes[1].bar(x + i*w, mdata['Portfolio_Return'].values * 100, w, label=m, color=colors[m])

axes[1].set_xticks(x + w*1.5)
axes[1].set_xticklabels(sorted(df_results['Window'].unique()))
axes[1].set_ylabel('Quarterly Return (%)')
axes[1].set_title('Quarterly Returns by Window')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# AUC comparison across windows
fig, ax = plt.subplots(figsize=(10, 5))
for m in summary['Model']:
    mdata = df_metrics[df_metrics['Model'] == m].sort_values('Window')
    ax.plot(mdata['Window'], mdata['AUC_Val'], 'o-', label=m, color=colors[m], linewidth=2, markersize=7)

ax.set_title('AUC-ROC Across Walk-Forward Windows')
ax.set_ylabel('AUC-ROC')
ax.axhline(0.5, color='red', linestyle='--', linewidth=1, label='Random Baseline')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. 特征重要性分析

In [ ]:
# Feature importance from XGBoost (last window)
if hasattr(m_xgb, 'feature_importances_'):
    imp = pd.DataFrame({
        'Feature': FEATURE_COLS,
        'Importance': m_xgb.feature_importances_
    }).sort_values('Importance', ascending=False).head(15)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(range(len(imp)), imp['Importance'].values, color='#7F77DD')
    ax.set_yticks(range(len(imp)))
    ax.set_yticklabels(imp['Feature'].values)
    ax.invert_yaxis()
    ax.set_title('XGBoost Top 15 Feature Importance (W5)')
    plt.tight_layout()
    plt.show()
else:
    print("XGBoost importance not available")

## Summary

This notebook implements a complete quantitative stock selection pipeline:

1. **Data**: 4,282 A-shares, 10 quarterly periods, fundamental data
2. **Features**: 31 engineered features (19 raw + 8 rank-normalized + 4 composite)
3. **Target**: Binary: P(Next_Ret > 0) for Kelly-compatible position sizing
4. **Models**: Logistic Regression / Random Forest / XGBoost / LightGBM
5. **Backtest**: 5-window Walk-Forward with user-configurable strategy parameters
6. **Evaluation**: Cumulative return, Sharpe, MaxDD, Calmar, Win Rate, AUC

The strategy layer (T_buy/T_sell/alpha/cap_stock/N_max/beta) is fully separated from
the model layer, allowing users to customize risk parameters without retraining.